# Round 1 parameter tuning notebook

This notebook does **grid search on your `strategy.py` constants** by:
1. reading the current `strategy.py`
2. writing temporary strategy files with different parameter values
3. running your existing backtester
4. using **final total PnL** as the score
5. ranking all parameter combinations

It is set up for your current folder layout:

```text
round_1_fixed/
├── strategy.py
├── data/
└── backtest/
    ├── backtester.py
    └── datamodel.py
```

It does **not** change your original `strategy.py`.

In [1]:
from pathlib import Path
import itertools
import json
import math
import os
import re
import shutil
import subprocess
import sys
import tempfile

import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# --- SET THIS TO YOUR PROJECT ROOT ---
# Example:
# PROJECT_ROOT = Path("/Users/yourname/.../round_1_fixed")

PROJECT_ROOT = Path.cwd()

BACKTEST_DIR = PROJECT_ROOT / "backtest"
BACKTESTER_PATH = BACKTEST_DIR / "backtester.py"
STRATEGY_PATH = PROJECT_ROOT / "strategy.py"
DATA_DIR = PROJECT_ROOT / "data"

ROUND_NUMBER = 1
DAYS = [-2, -1, 0]
CARRY_STATE_ACROSS_DAYS = True  # matches your current setup more closely

assert PROJECT_ROOT.exists(), f"PROJECT_ROOT not found: {PROJECT_ROOT}"
assert BACKTESTER_PATH.exists(), f"Missing backtester.py: {BACKTESTER_PATH}"
assert STRATEGY_PATH.exists(), f"Missing strategy.py: {STRATEGY_PATH}"
assert DATA_DIR.exists(), f"Missing data dir: {DATA_DIR}"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("BACKTESTER_PATH:", BACKTESTER_PATH)
print("STRATEGY_PATH:", STRATEGY_PATH)
print("DATA_DIR:", DATA_DIR)

PROJECT_ROOT: /Users/wuyuming/Downloads/IMC_GoTritons/round_1_fixed
BACKTESTER_PATH: /Users/wuyuming/Downloads/IMC_GoTritons/round_1_fixed/backtest/backtester.py
STRATEGY_PATH: /Users/wuyuming/Downloads/IMC_GoTritons/round_1_fixed/strategy.py
DATA_DIR: /Users/wuyuming/Downloads/IMC_GoTritons/round_1_fixed/data


In [3]:
base_source = STRATEGY_PATH.read_text()

print("Loaded strategy.py")
print("-" * 80)
print("\n".join(base_source.splitlines()[:60]))

Loaded strategy.py
--------------------------------------------------------------------------------
from datamodel import OrderDepth, TradingState, Order
from typing import Dict, List
import json
import math


class Trader:
    ASH_PRODUCT = "ASH_COATED_OSMIUM"

    POSITION_LIMITS = {
        ASH_PRODUCT: 80,
    }

    ASH_HISTORY_LENGTH = 3
    ASH_ENTRY_Z = 1.0
    ASH_EXIT_Z = 0.4
    ASH_BASE_QUOTE_OFFSET = 1
    ASH_INVENTORY_SKEW = 0.02

    def __init__(self):
        self.position = 0
        self.mid_history = []

    def bid(self):
        return 15

    def send_sell_order(self, orders, product, price, amount):
        if amount != 0:
            orders.append(Order(product, int(price), int(amount)))

    def send_buy_order(self, orders, product, price, amount):
        if amount != 0:
            orders.append(Order(product, int(price), int(amount)))

    def get_product_pos(self, state, product):
        return state.position.get(product, 0)

    def load_saved_data(self

In [4]:
def replace_class_constant(source: str, name: str, value) -> str:
    """
    Replace a line like:
        ASH_ENTRY_Z = 1.0
    with:
        ASH_ENTRY_Z = <value_repr>
    """
    pattern = rf"(^\s*{re.escape(name)}\s*=\s*)(.+)$"
    repl = rf"\g<1>{repr(value)}"
    new_source, n = re.subn(pattern, repl, source, count=1, flags=re.MULTILINE)
    if n == 0:
        raise ValueError(f"Could not find constant: {name}")
    return new_source


def replace_mm_sizes(
    source: str, flat_size: int, mid_size: int, large_size: int
) -> str:
    """
    Replace the hardcoded returns inside get_mm_size():
        if a < 20:
            return 10
        elif a < 50:
            return 6
        else:
            return 3
    """
    pattern = (
        r"(def\s+get_mm_size\(self,\s*position:\s*int\)\s*->\s*int:\s*\n"
        r"\s*a\s*=\s*abs\(position\)\s*\n"
        r"\s*if\s+a\s*<\s*20:\s*\n\s*return\s+)(\d+)"
        r"(\s*\n\s*elif\s+a\s*<\s*50:\s*\n\s*return\s+)(\d+)"
        r"(\s*\n\s*else:\s*\n\s*return\s+)(\d+)"
    )

    repl = rf"\g<1>{flat_size}\g<3>{mid_size}\g<5>{large_size}"
    new_source, n = re.subn(pattern, repl, source, count=1, flags=re.MULTILINE)
    if n == 0:
        raise ValueError("Could not replace get_mm_size tiers.")
    return new_source


def build_strategy_source(
    base_source: str,
    ash_history_length: int,
    ash_entry_z: float,
    ash_base_quote_offset: int,
    ash_inventory_skew: float,
    mm_flat: int,
    mm_mid: int,
    mm_large: int,
) -> str:
    src = base_source
    src = replace_class_constant(src, "ASH_HISTORY_LENGTH", ash_history_length)
    src = replace_class_constant(src, "ASH_ENTRY_Z", ash_entry_z)
    src = replace_class_constant(
        src, "ASH_BASE_QUOTE_OFFSET", ash_base_quote_offset
    )
    src = replace_class_constant(src, "ASH_INVENTORY_SKEW", ash_inventory_skew)
    src = replace_mm_sizes(src, mm_flat, mm_mid, mm_large)
    return src

In [5]:
def run_one_backtest(
    strategy_source: str,
    project_root: Path,
    round_number: int = 1,
    days=(-2, -1, 0),
    carry_state_across_days: bool = True,
):
    """
    Writes a temporary strategy file and runs the provided backtester.
    Score = final total pnl from summary.csv for day == 'ALL'
    """
    backtester_path = project_root / "backtest" / "backtester.py"
    data_dir = project_root / "data"

    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir = Path(tmpdir)
        strategy_path = tmpdir / "strategy.py"
        output_dir = tmpdir / "backtest_results"

        strategy_path.write_text(strategy_source)

        cmd = [
            sys.executable,
            str(backtester_path),
            "--strategy",
            str(strategy_path),
            "--data-dir",
            str(data_dir),
            "--round-number",
            str(round_number),
            "--days",
            *[str(d) for d in days],
            "--output-dir",
            str(output_dir),
        ]

        if carry_state_across_days:
            cmd.append("--carry-state-across-days")

        completed = subprocess.run(
            cmd,
            cwd=str(project_root),
            capture_output=True,
            text=True,
        )

        if completed.returncode != 0:
            return {
                "ok": False,
                "final_total_pnl": float("-inf"),
                "stdout": completed.stdout,
                "stderr": completed.stderr,
                "cmd": cmd,
            }

        summary_path = output_dir / "summary.csv"
        pnl_path = output_dir / "pnl_timeseries.csv"
        fills_path = output_dir / "fills.csv"

        summary_df = pd.read_csv(summary_path)
        all_row = summary_df.loc[summary_df["day"].astype(str) == "ALL"].iloc[
            0
        ]
        final_total_pnl = float(all_row["final_total_pnl"])

        return {
            "ok": True,
            "final_total_pnl": final_total_pnl,
            "summary_df": summary_df,
            "pnl_df": pd.read_csv(pnl_path),
            "fills_df": pd.read_csv(fills_path),
            "stdout": completed.stdout,
            "stderr": completed.stderr,
            "cmd": cmd,
        }

In [6]:
# --- SMALL LOCAL SEARCH AROUND YOUR CURRENT BEST ---
# Start small. Expand only after you confirm the notebook works.

param_grid = {
    "ash_history_length": [3],
    "ash_entry_z": [0.9, 1.0, 1.1],
    "ash_base_quote_offset": [0, 1, 2],
    "ash_inventory_skew": [0.01, 0.02, 0.03],
    "mm_flat": [10, 12],
    "mm_mid": [6, 8],
    "mm_large": [3, 4],
}

grid_keys = list(param_grid.keys())
all_combos = [
    dict(zip(grid_keys, values))
    for values in itertools.product(*(param_grid[k] for k in grid_keys))
]

print(f"Number of combinations: {len(all_combos)}")
all_combos[:5]

Number of combinations: 216


[{'ash_history_length': 3,
  'ash_entry_z': 0.9,
  'ash_base_quote_offset': 0,
  'ash_inventory_skew': 0.01,
  'mm_flat': 10,
  'mm_mid': 6,
  'mm_large': 3},
 {'ash_history_length': 3,
  'ash_entry_z': 0.9,
  'ash_base_quote_offset': 0,
  'ash_inventory_skew': 0.01,
  'mm_flat': 10,
  'mm_mid': 6,
  'mm_large': 4},
 {'ash_history_length': 3,
  'ash_entry_z': 0.9,
  'ash_base_quote_offset': 0,
  'ash_inventory_skew': 0.01,
  'mm_flat': 10,
  'mm_mid': 8,
  'mm_large': 3},
 {'ash_history_length': 3,
  'ash_entry_z': 0.9,
  'ash_base_quote_offset': 0,
  'ash_inventory_skew': 0.01,
  'mm_flat': 10,
  'mm_mid': 8,
  'mm_large': 4},
 {'ash_history_length': 3,
  'ash_entry_z': 0.9,
  'ash_base_quote_offset': 0,
  'ash_inventory_skew': 0.01,
  'mm_flat': 12,
  'mm_mid': 6,
  'mm_large': 3}]

In [7]:
results = []

for i, params in enumerate(all_combos, start=1):
    strategy_source = build_strategy_source(
        base_source=base_source,
        ash_history_length=params["ash_history_length"],
        ash_entry_z=params["ash_entry_z"],
        ash_base_quote_offset=params["ash_base_quote_offset"],
        ash_inventory_skew=params["ash_inventory_skew"],
        mm_flat=params["mm_flat"],
        mm_mid=params["mm_mid"],
        mm_large=params["mm_large"],
    )

    out = run_one_backtest(
        strategy_source=strategy_source,
        project_root=PROJECT_ROOT,
        round_number=ROUND_NUMBER,
        days=DAYS,
        carry_state_across_days=CARRY_STATE_ACROSS_DAYS,
    )

    row = dict(params)
    row["ok"] = out["ok"]
    row["final_total_pnl"] = out["final_total_pnl"]

    if not out["ok"]:
        row["stderr"] = out["stderr"][:1000]
        print(f"[{i}/{len(all_combos)}] FAILED:", params)
    else:
        print(
            f"[{i}/{len(all_combos)}] pnl={out['final_total_pnl']:.2f} | {params}"
        )

    results.append(row)

results_df = (
    pd.DataFrame(results)
    .sort_values("final_total_pnl", ascending=False)
    .reset_index(drop=True)
)
results_df.head(20)

[1/216] pnl=6192.00 | {'ash_history_length': 3, 'ash_entry_z': 0.9, 'ash_base_quote_offset': 0, 'ash_inventory_skew': 0.01, 'mm_flat': 10, 'mm_mid': 6, 'mm_large': 3}
[2/216] pnl=6192.00 | {'ash_history_length': 3, 'ash_entry_z': 0.9, 'ash_base_quote_offset': 0, 'ash_inventory_skew': 0.01, 'mm_flat': 10, 'mm_mid': 6, 'mm_large': 4}
[3/216] pnl=6192.00 | {'ash_history_length': 3, 'ash_entry_z': 0.9, 'ash_base_quote_offset': 0, 'ash_inventory_skew': 0.01, 'mm_flat': 10, 'mm_mid': 8, 'mm_large': 3}
[4/216] pnl=6192.00 | {'ash_history_length': 3, 'ash_entry_z': 0.9, 'ash_base_quote_offset': 0, 'ash_inventory_skew': 0.01, 'mm_flat': 10, 'mm_mid': 8, 'mm_large': 4}
[5/216] pnl=6192.00 | {'ash_history_length': 3, 'ash_entry_z': 0.9, 'ash_base_quote_offset': 0, 'ash_inventory_skew': 0.01, 'mm_flat': 12, 'mm_mid': 6, 'mm_large': 3}
[6/216] pnl=6192.00 | {'ash_history_length': 3, 'ash_entry_z': 0.9, 'ash_base_quote_offset': 0, 'ash_inventory_skew': 0.01, 'mm_flat': 12, 'mm_mid': 6, 'mm_large': 4

KeyboardInterrupt: 

In [12]:
# Pick two absurdly different parameter sets
params_a = {
    "ash_history_length": 3,
    "ash_entry_z": 0.2,
    "ash_base_quote_offset": 0,
    "ash_inventory_skew": 0.0,
    "mm_flat": 12,
    "mm_mid": 8,
    "mm_large": 4,
}

params_b = {
    "ash_history_length": 3,
    "ash_entry_z": 5.0,
    "ash_base_quote_offset": 5,
    "ash_inventory_skew": 1.0,
    "mm_flat": 1,
    "mm_mid": 1,
    "mm_large": 1,
}

src_a = build_strategy_source(base_source=base_source, **params_a)
src_b = build_strategy_source(base_source=base_source, **params_b)

print("Sources identical?", src_a == src_b)

# print only the parameter lines
for label, src in [("A", src_a), ("B", src_b)]:
    print("\n" + "=" * 60)
    print("SOURCE", label)
    for line in src.splitlines():
        if any(
            k in line
            for k in [
                "ASH_HISTORY_LENGTH",
                "ASH_ENTRY_Z",
                "ASH_BASE_QUOTE_OFFSET",
                "ASH_INVENTORY_SKEW",
                "return 10",
                "return 6",
                "return 3",
                "return 12",
                "return 8",
                "return 4",
                "return 1",
                "def get_mm_size",
            ]
        ):
            print(line)

Sources identical? False

SOURCE A
    ASH_HISTORY_LENGTH = 3
    ASH_ENTRY_Z = 0.2
    ASH_BASE_QUOTE_OFFSET = 0
    ASH_INVENTORY_SKEW = 0.0
        return 15
                    "mid_history": self.mid_history[-self.ASH_HISTORY_LENGTH :]
        if len(self.mid_history) > self.ASH_HISTORY_LENGTH:
    def get_mm_size(self, position: int) -> int:
            return 12
            return 8
            return 4
        if z < -self.ASH_ENTRY_Z:
        elif z > self.ASH_ENTRY_Z:
        fair = mean - self.ASH_INVENTORY_SKEW * est_pos
        bid_price = min(best_bid + 1, int(fair - self.ASH_BASE_QUOTE_OFFSET))
        ask_price = max(best_ask - 1, int(fair + self.ASH_BASE_QUOTE_OFFSET))

SOURCE B
    ASH_HISTORY_LENGTH = 3
    ASH_ENTRY_Z = 5.0
    ASH_BASE_QUOTE_OFFSET = 5
    ASH_INVENTORY_SKEW = 1.0
        return 15
                    "mid_history": self.mid_history[-self.ASH_HISTORY_LENGTH :]
        if len(self.mid_history) > self.ASH_HISTORY_LENGTH:
    def get_mm_size(self, pos

In [13]:
out_a = run_one_backtest(
    strategy_source=src_a,
    project_root=PROJECT_ROOT,
    round_number=ROUND_NUMBER,
    days=DAYS,
    carry_state_across_days=CARRY_STATE_ACROSS_DAYS,
)

out_b = run_one_backtest(
    strategy_source=src_b,
    project_root=PROJECT_ROOT,
    round_number=ROUND_NUMBER,
    days=DAYS,
    carry_state_across_days=CARRY_STATE_ACROSS_DAYS,
)

print("PNL A:", out_a["final_total_pnl"])
print("PNL B:", out_b["final_total_pnl"])
print("OK A:", out_a["ok"])
print("OK B:", out_b["ok"])
print("\nSTDOUT A:\n", out_a["stdout"][:1000])
print("\nSTDOUT B:\n", out_b["stdout"][:1000])
print("\nSTDERR A:\n", out_a["stderr"][:1000])
print("\nSTDERR B:\n", out_b["stderr"][:1000])

EmptyDataError: No columns to parse from file

In [ ]:
# Save tuning results
results_path = PROJECT_ROOT / "tuning_results.csv"
results_df.to_csv(results_path, index=False)
print("Saved:", results_path)

In [ ]:
# Top parameter sets
top_n = 20
display(results_df.head(top_n))

In [ ]:
# Inspect how each parameter affects pnl on average
for col in ["ash_entry_z", "ash_base_quote_offset", "ash_inventory_skew", "mm_flat", "mm_mid", "mm_large"]:
    print("\n", "=" * 80)
    print(col)
    print(results_df.groupby(col)["final_total_pnl"].agg(["count", "mean", "max"]).sort_values("max", ascending=False))

In [ ]:
# Re-run the single best configuration and plot its PnL
best_params = results_df.iloc[0].to_dict()
best_params

In [ ]:
best_source = build_strategy_source(
    base_source=base_source,
    ash_history_length=int(best_params["ash_history_length"]),
    ash_entry_z=float(best_params["ash_entry_z"]),
    ash_base_quote_offset=int(best_params["ash_base_quote_offset"]),
    ash_inventory_skew=float(best_params["ash_inventory_skew"]),
    mm_flat=int(best_params["mm_flat"]),
    mm_mid=int(best_params["mm_mid"]),
    mm_large=int(best_params["mm_large"]),
)

best_run = run_one_backtest(
    strategy_source=best_source,
    project_root=PROJECT_ROOT,
    round_number=ROUND_NUMBER,
    days=DAYS,
    carry_state_across_days=CARRY_STATE_ACROSS_DAYS,
)

best_pnl_df = best_run["pnl_df"].copy()

# global time for plotting across multiple days
best_pnl_df = best_pnl_df.sort_values(["day", "timestamp"]).copy()
min_day = int(best_pnl_df["day"].min())
day_span = int(best_pnl_df["timestamp"].max()) + 1
best_pnl_df["global_time"] = (best_pnl_df["day"] - min_day) * day_span + best_pnl_df["timestamp"]

plt.figure(figsize=(12, 5))
plt.plot(best_pnl_df["global_time"], best_pnl_df["total_pnl"])
plt.xlabel("Global Time")
plt.ylabel("Total PnL")
plt.title("Best Parameter Set: PnL Over Time")
plt.grid(True)
plt.tight_layout()
plt.show()

display(best_run["summary_df"])

In [ ]:
# Optional: write the best strategy to a file you can test directly
best_strategy_path = PROJECT_ROOT / "strategy_best_from_tuning.py"
best_strategy_path.write_text(best_source)
print("Saved best strategy to:", best_strategy_path)

## Faster next pass

Once this works, do **local search** instead of huge grids.

Example next grid:
- keep the best `mm_flat`, `mm_mid`, `mm_large`
- only tune:
  - `ash_entry_z`
  - `ash_base_quote_offset`
  - `ash_inventory_skew`

That is usually better than searching everything at once.